# Module C, sequence 3: analysis (corrected version)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pymodaq_utils.math_utils import gauss1D
from pymodaq_data.h5modules.data_saving import DataLoader

DATA = Path('../data')

## 1. Load the scan file (Scan000: one scan, Scan001: scan_average = 8)

In [ ]:
loader = DataLoader(DATA / 'scan_photodiode.h5')
scan = loader.load_data('/RawData/Scan000/Detector000/Data0D/CH00/Data00', load_all=True)
scan8 = loader.load_data('/RawData/Scan001/Detector000/Data0D/CH00/Data00', load_all=True)
loader.close_file()
print(scan, scan.axes)
print(scan8, scan8.axes)

## 2. Look at it and find the lines

In [ ]:
mean8 = scan8.mean(0)  # mean over the first navigation axis (the repetitions)
wavelength = mean8.get_axis_from_index(0)[0].get_data()
fig, ax = plt.subplots()
ax.plot(wavelength, scan.data[0], '.', label='one scan')
ax.plot(wavelength, mean8.data[0], '-', label='mean of 8 scans')
ax.set_xlabel('wavelength (nm)')
ax.legend()
peaks, _ = find_peaks(mean8.data[0], height=1.0)
print(wavelength[peaks])

## 3. Crop around the main line and fit with scipy (uncertainties from the covariance)

In [ ]:
def model(x, amp, x0, fwhm, offset):
    return amp * gauss1D(x, x0, fwhm) + offset


window = mean8.vnav[505.:555.]
window.nav_indexes = ()  # the navigation axis becomes the signal axis: a 1D data
x = window.get_axis_from_index(0)[0].get_data()
y = window.data[0]
p0 = [y.max() - y.min(), x[np.argmax(y)], 5., y.min()]
popt, pcov = curve_fit(model, x, y, p0=p0)
perr = np.sqrt(np.diag(pcov))
for name, value, error in zip(['amplitude', 'center (nm)', 'FWHM (nm)', 'offset'], popt, perr):
    print(f'{name}: {value:.3f} +/- {error:.3f}')

## 4. The same fit with DataWithAxes.fit (no uncertainties returned)

In [ ]:
fit = window.fit(model, p0)
print(fit.fit_coeffs[0])
assert np.allclose(fit.fit_coeffs[0], popt, rtol=1e-3)

## 5. Uncertainties from the repetitions, weighted fit

In [ ]:
n_repeat = scan8.shape[0]
mean8.errors = [scan8.data[0].std(axis=0, ddof=1) / np.sqrt(n_repeat)]  # standard error of the mean
window = mean8.vnav[505.:555.]  # the errors follow the slicing
window.nav_indexes = ()
popt_w, pcov_w = curve_fit(model, x, window.data[0], p0=p0, sigma=window.errors[0], absolute_sigma=True)
chi2_red = np.sum(((window.data[0] - model(x, *popt_w)) / window.errors[0]) ** 2) / (len(x) - len(popt_w))
print(popt_w, np.sqrt(np.diag(pcov_w)))
print('reduced chi2:', chi2_red)  # larger than 1: the intensity drift is not in the error bars

## 6. The axis in eV

In [ ]:
axis_ev = window.get_axis_from_index(0)[0].units_as('eV', inplace=False, context='spectroscopy')
window_ev = window.deepcopy()
window_ev.axes[0] = axis_ev
x_ev = axis_ev.get_data()
y_ev = window_ev.data[0]
p0_ev = [10., x_ev[np.argmax(y_ev)], 0.02, y_ev.min()]
popt_ev, pcov_ev = curve_fit(model, x_ev, y_ev, p0=p0_ev)
print(axis_ev.units, popt_ev[1], popt_ev[2], np.sqrt(np.diag(pcov_ev))[1:3])

## 7. One spectrum with Naverage = 200: the errors come with the data (PyMoDAQ 5.3.9 or later)

In [ ]:
loader = DataLoader(DATA / 'spectrum_naverage200.h5')
spectrum = loader.load_all('/RawData/Detector000').get_data_from_name('Spectrum')
loader.close_file()
print(spectrum.errors[0][:3])

## 8. Compare with the standard deviation of the continuous recording

In [ ]:
loader = DataLoader(DATA / 'continuous_spectra.h5')
series = loader.load_data('/RawData/Detector000/Data1D/CH00/EnlData00', load_all=True)
loader.close_file()
n_spectra = series.shape[0]
sem_series = series.data[0].std(axis=0, ddof=1) / np.sqrt(n_spectra)
i_peak = int(np.argmax(spectrum.data[0]))
print('standard error at the peak, Naverage=200:', spectrum.errors[0][i_peak], ' series:', sem_series[i_peak])
print('standard error far from the peak, Naverage=200:', spectrum.errors[0][:10].mean(), ' series:', sem_series[:10].mean())
fig, ax = plt.subplots()
ax.plot(series.data[0][:, i_peak], '.-')
ax.set_xlabel('acquisition number')
ax.set_ylabel('intensity at the peak')